# Design optimization of coronagraph for  1D geometry
This notebook shows an example of coronagraph design optimization for 1D geometry with the corono package.

Author: Mamadou N'Diaye <mamadou.ndiaye@oca.eu> (https://github.com/astromam)

License: MIT license

## Initialization
This section lists all the required packages to run the notebook.

In [ ]:
import numpy as np
import time
import os
import pylab as pl

from pathlib import Path
import corono as coro

## Parameters
This section lists all the parameters for the coronagraph and the optimization problem.

### Solving parameters

Coronagraph and Optimization problem

In [ ]:
# coronagraph, problem, and solver types 
corono_name  = 'APLC'   # 'APLC' or 'SP'
problem_name = 'MaxTau' # 'MaxTau' # 'MaxContrastLinf' #'MaxContrastL1'
solver       = 'scipy.linprog' # 'stdgrb', 'gurobipy', 'scipy.linprog'

# keywords for solver
slvLogToConsole = 0
slvCrossover    = 0
slvMethod       = 2
allLogToConsole = 0

# additional constraints and their parameters
FirstDer    = False
SecondDer   = False
MinIsland   = False
Binarity    = False
FirstDerLim       = 0.01
SecondDerLim      = 0.001 
FirstDerGlobalLim = 10.
BinarityReg       = 0.000000001

Sampling

In [ ]:
nPup = 500
nFPM = 50
nImg = 44
Fmax = 11

Optical system

In [ ]:
# spectral bandwidth
bw   = 0.1
nlam = 5

# Pupil inner diameter (ID) and radius
PupilID    = 0.20

# Focal plane mask in lam0/D unit 
rMask       = 4.4

# Lyot stop inner and outer diameter (ID and OD) in fraction of pupil diameter
LyotStopID = 0.40
LyotStopOD = 1.0

Optimization

In [ ]:
# dark zone bounds (inner and outer edges) in lam0/D unit
rho0 = 3.5
rho1 = 10.0

# contrast in the dark region
cDarkHole = 8.0

# tau (integrated Pupil transmission)
tau   = 0.3

Pupils

In [ ]:
# radius in R unit.
R            = 1
r            = np.arange(nPup)*R/nPup + R/(2*nPup)

# Pupil definition
Pupil1d      = (r>PupilID)*1.0
LyotStop1d   = (r>LyotStopID)*(r<LyotStopOD)*1.0

Dictionary

In [ ]:
# Test on solver type
if solver != 'gurobipy' and solver != 'stdgrb':
    solver = 'scipy'

# Definition of a dictionary of parameters 
params = coro.to_dict(rho0=rho0, rho1=rho1, cDarkHole=cDarkHole, tau=tau,
                 nPup = nPup, nFPM=nFPM, nImg=nImg, Fmax = Fmax,
                 bw = bw, nlam = nlam,
                 PupilID = PupilID, rMask = rMask, 
                 LyotStopID = LyotStopID,
                 LyotStopOD = LyotStopOD,
                 r = r, R=R, Pupil1d = Pupil1d, LyotStop1d = LyotStop1d,
                 solver = solver, problem_name = problem_name,
                 corono_name = corono_name, slvLogToConsole = slvLogToConsole,
                 slvCrossover = slvCrossover, slvMethod = slvMethod,
                 allLogToConsole = allLogToConsole,
                 FirstDer = FirstDer, SecondDer = SecondDer,
                 FirstDerLim = FirstDerLim, SecondDerLim = SecondDerLim,
                 MinIsland = MinIsland, FirstDerGlobalLim = FirstDerGlobalLim,
                 Binarity = Binarity, BinarityReg = BinarityReg)


### Plot parameters

In [ ]:
# spectral sampling
nlambis = 11

# image sampling
nImgbis = 110

# maximum spatial frequency in the image
Fmaxbis = 11

## Coronagraph definition
Definition of an object from the coronagraph class in design module.

In [ ]:
if corono_name == 'APLC':
    # Apodized Pupil Lyot Coronagraph
    corono0 = coro.design.APLC1d(**params)
elif corono_name == 'SP':
    # Shaped Pupil
    corono0 = coro.design.SP1d(**params)
else:
    raise NameError('{0}: Not an existing coronagraph!'.format(corono_name))

## Problem definition
Definition of an object from the optim_1d class in optim_1d module.

In [ ]:
if problem_name == 'MaxTau':
    # Maximization of the integrated amplitude transmission of the apodizer
    problem1 = coro.optim_1d.MaxTau(corono=corono0, **params)
elif problem_name == 'MaxContrastL1':
    # Maximization of the contrast under L1-norm
    problem1 = coro.optim_1d.MaxContrast(corono=corono0, Lnorm='L1',**params)
elif problem_name == 'MaxContrastLinf':
    # Maximization of the contrast under L-infinite norm
    problem1 = coro.optim_1d.MaxContrast(corono=corono0, Lnorm='Linf',**params)
else:
     raise NameError('{0}: Not an existing optimization problem!'.format(problem_name))

## Problem solving

In [ ]:
# Problem solving 
t0 = time.time()
Apod = problem1.solve_model()
t1 = time.time()
print('optimization time              : {0:.2f}s'.format(t1-t0))

# Apodizer normalization
norm_apod = 1./Apod.max()
Apod *= norm_apod

## Plot display of the amplitude transmissions of the optical system elements

### Aperture

In [ ]:
pl.figure(1)
pl.clf()
pl.title('Aperture')
pl.plot(corono0.r, Pupil1d)
pl.xlabel(r'Pupil radius r')
pl.ylabel('Pupil amplitude transmission')
pl.tight_layout()
pl.show()


### Lyot Stop

In [ ]:
pl.figure(2)
pl.clf()
pl.title('Lyot stop')
pl.plot(corono0.r, LyotStop1d)
pl.xlabel(r'Pupil radius r')
pl.ylabel('Lyot Stop amplitude transmission')
pl.tight_layout()
pl.show()

### Apodizer

In [ ]:
pl.figure(3)
pl.clf()
pl.title('Apodizer solution found with {0}'.format(solver))
pl.plot(corono0.r, Apod, label=solver)
pl.xlabel(r'Pupil radius r')
pl.ylabel('Apodizer amplitude transmission')
pl.tight_layout()
pl.show()

## Computation of the direct and coronagraphic images

Update of the params dictionary for plot purpose

In [ ]:
params2    = coro.update_params(params, nlam=nlambis, nImg=nImgbis, Fmax=Fmaxbis) 

Definition of the corono.design with updated parameters

In [ ]:
if corono_name == 'APLC':
    # Apodized Pupil Lyot Coronagraph
    corono0 = coro.design.APLC1d(**params2)
elif corono_name == 'SP':
    # Shaped Pupil
    corono0 = coro.design.SP1d(**params2)
else:
    raise NameError('{0}: Not an existing coronagraph!'.format(corono_name))

Computation of the broadband light images

In [ ]:
# broadband light image computation with and without coronagraph mask
poly_direct_image1 = corono0.compute_direct_intensity_1d(Apod)
poly_corono_image1 = corono0.compute_corono_intensity_1d(Apod)

# image normalization
peak_norm_poly = 1./poly_direct_image1.max()
poly_direct_image1 *= peak_norm_poly
poly_corono_image1 *= peak_norm_poly

Computation of the monochromatic light images

In [ ]:
# monochromatic light image computation with and without coronagraph mask
mono_direct_image1 = corono0.compute_direct_intensity_1d(Apod, poly=False)
mono_corono_image1 = corono0.compute_corono_intensity_1d(Apod, poly=False)

# image normalization
peak_norm_mono = 1./mono_direct_image1[(corono0.nlam+1)//2].max()
mono_direct_image1 *= peak_norm_mono
mono_corono_image1 *= peak_norm_mono

## Plot display of the intensity profiles of the coronagraphic images

###  Broadband light

In [ ]:
pl.figure(2)
pl.clf()
pl.title('Intensity profiles in broadband light')
# Direct image
#pl.semilogy(corono0.xi,poly_direct_image1,label='Direct')
# Coronagraphic image
pl.semilogy(corono0.xi,poly_corono_image1,label='Corono')
pl.axvline(x=corono0.rMask, ymin=-12, ymax =2, linewidth=1, color='r', linestyle='--')
pl.axvline(x=corono0.rho0, ymin=-12, ymax =2, linewidth=1, color='b', linestyle='--')
pl.axvline(x=corono0.rho1, ymin=-12, ymax =2, linewidth=1, color='b', linestyle='--')
pl.axhline(10**(-cDarkHole), xmin=corono0.xi.min(), xmax=corono0.xi.max(), linewidth=1, color='k', linestyle='--')
pl.xlabel(r'Angular separation in $\lambda_0$/D')
pl.ylabel('Normalized intensity in log scale')
pl.ylim(1e-12, 1e-3)
pl.legend()
pl.tight_layout()
pl.show()

### Monochromatic light

In [ ]:
values = range(nlambis)
colors = pl.cm.rainbow(np.linspace(0,1,nlambis))

pl.figure(3)
pl.clf()
pl.title('Intensity profiles in monochromatic light')
for i in range(corono0.nlam):
    # direct image
#     pl.semilogy(corono0.xi,mono_direct_image1[i], 
#                 label=r'{0:.2f}$\lambda_0$'.format(corono0.lam_t[i]), 
#                 color = colors[i])
    # coronagraphic image
    pl.semilogy(corono0.xi,mono_corono_image1[i], 
                label=r'{0:.2f}$\lambda_0$'.format(corono0.lam_t[i]), 
                color = colors[i])
pl.axvline(x=corono0.rMask, ymin=-12, ymax =2, linewidth=1, color='r', linestyle='--')
pl.axvline(x=corono0.rho0, ymin=-12, ymax =2, linewidth=1, color='b', linestyle='--')
pl.axvline(x=corono0.rho1, ymin=-12, ymax =2, linewidth=1, color='b', linestyle='--')
pl.axhline(10**(-cDarkHole), xmin=corono0.xi.min(), xmax=corono0.xi.max(), linewidth=1, color='k', linestyle='--')
pl.xlabel(r'Angular separation in $\lambda_0$/D')
pl.ylabel('Normalized intensity in log scale')
pl.ylim(1e-12, 1e-3)
pl.legend()
pl.tight_layout()

pl.show()